# Project Evals — Continue / Schedule / End

Use the **OpenAI Evals API** to test how well a prompted model classifies the next recruiter action on our labeled `sms_conversations.json` data.

This mirrors **Part A** of the class notebook, but the task is our project's task (not e-commerce complaints).

At the end there is also a **bonus section** that evaluates our actual multi-agent system (`get_main_agent_response`) locally.

## Setup

### Install dependencies (run once)
```powershell
pip install openai python-dotenv scikit-learn pandas matplotlib seaborn
```

In [ ]:
# Import the OpenAI client, the dotenv loader, and os for environment variables.
from openai import OpenAI
from dotenv import load_dotenv
import os

In [ ]:
# Load environment variables from .env and read the OpenAI API key.
# Print only the first few characters to confirm it loaded (never the full key).

load_dotenv()  # Loads variables from .env

openai_key = os.getenv("OPENAI_API_KEY")
print(openai_key[:5])  # Just to check, don't print the full key!

In [ ]:
# Initialize the OpenAI client (reads OPENAI_API_KEY from the environment).
client = OpenAI()

## Prepare the test data

Convert `Relevant Files/sms_conversations.json` into a **JSONL** file where each line is one test item. We classify the next recruiter action given the conversation so far.

Each line will look like:
```json
{ "item": { "conversation_history": "recruiter: ...\ncandidate: ...", "correct_label": "continue" } }
```

In [ ]:
# Load the labeled SMS conversations dataset into a list of conversation dicts.
import json

# Open and load the conversations JSON file:
with open("../Relevant Files/sms_conversations.json", encoding="utf-8") as conversations_file:
    conversations = json.load(conversations_file)

print(json.dumps(conversations[0], indent=4))

In [ ]:
# Helper that turns a list of conversation turns into a single readable string,
# one "speaker: text" line per turn. Used to build each test case's history.

def format_history(turns):
    return "\n".join(
        f"{turn['speaker']}: {turn['text']}"
        for turn in turns
        )

In [25]:
print(format_history(conversations[0]["turns"]))

recruiter: Thanks for applying to our Python Developer opening. What kinds of Python projects have you worked on recently?
candidate: I've been using Python professionally for five years, mostly for data analysis.
recruiter: Our engineering manager can interview you Wednesday at 10 AM or Thursday at 2 PM. Which works best?
candidate: I can't at that time—I'm busy.
recruiter: No problem. How about Thursday at 4 PM instead?
candidate: Monday at 3 PM is good.
recruiter: Great, your interview is confirmed. You'll receive a calendar invite shortly.


In [ ]:
# Build the list of test cases: for every labeled recruiter turn, pair the
# conversation history up to that point with the correct next-action label.
# Candidate turns have label=None and are skipped.

test_items = []

for conversation in conversations:
    turns = conversation["turns"]

    for turn_index, turn in enumerate(turns):
        speaker = turn["speaker"]
        label = turn["label"] # Not null if recruiter

        if speaker == "recruiter" and label is not None:
            prior_turns = turns[:turn_index] # Everything BEFORE this turn
            history = format_history(prior_turns)
            test_items.append({'conversation_history': history, 'correct_label': label}) 


In [ ]:
# Write the test cases to JSONL, one {"item": {...}} object per line
# (the format the OpenAI Evals API expects).

with open("sms_conversations_test.jsonl", "w" , encoding="utf-8") as test_file:
    for item in test_items:
        json.dump({"item": item}, test_file)
        test_file.write("\n")

## Define the task

The instructions tell the model what to output. We want a single word: `continue`, `schedule`, or `end`.

In [ ]:
# System prompt: instruct the model to classify the next recruiter action
# as exactly one of 'continue', 'schedule', or 'end' and reply with only that word.
instructions = """
You are a classifier that labels the next recruiter action in an SMS recruitment
chatbot for a Python Developer position.

Given a conversation history between a recruiter and a candidate, decide what the
recruiter should do NEXT, and output exactly one of three labels:

    continue  — Keep the conversation going. Use when the candidate is still sharing
                background, asking questions about the role, or the conversation is
                in its early stages and there is not yet enough context to schedule
                an interview. Also use when the candidate cannot make a proposed
                time slot and an alternative is needed but none has been offered yet.

    schedule  — The recruiter should propose, validate, or confirm an interview slot.
                Use when the candidate has shown interest and shared enough background,
                the candidate brings up availability or specific dates, the recruiter
                is offering time slots, or an interview is being confirmed.

    end       — The conversation should be closed. Use when the candidate explicitly
                declines (not interested, already found a job, asks to stop), or when
                an interview has just been confirmed and there is nothing left to
                discuss.

OUTPUT RULES (strict — your output is graded by exact string match):
 - Reply with EXACTLY one word: continue, schedule, or end.
 - Lowercase only.
 - No punctuation, no quotes, no explanation, no whitespace before or after.

EXAMPLES:

  History:
  recruiter: Thanks for applying to our Python Developer opening. Could you tell me about your recent Python work?
  candidate: I've been using Python professionally for five years, mostly for data analysis and building internal
  tools.
  Answer: continue

  History:
  recruiter: Thanks for sharing your background — sounds like a great fit.
  candidate: Thanks! When would the interview be?
  Answer: schedule

  History:
  recruiter: Would you be available for an interview next week?
  candidate: Actually, I just accepted another offer last week, so I have to withdraw. Thanks anyway.
  Answer: end

Now classify the following conversation. Remember: reply with only one word.
"""

In [29]:
test_items[2]

{'conversation_history': "recruiter: Thanks for applying to our Python Developer opening. What kinds of Python projects have you worked on recently?\ncandidate: I've been using Python professionally for five years, mostly for data analysis.\nrecruiter: Our engineering manager can interview you Wednesday at 10\u202fAM or Thursday at 2\u202fPM. Which works best?\ncandidate: I can't at that time—I'm busy.",
 'correct_label': 'schedule'}

In [ ]:
# Quick sanity check: run one sample conversation through the model
# and print its reply to verify it returns one of the three labels.

completion = client.chat.completions.create(
    model="gpt-4.1",
    messages=[
        {"role": "developer", "content": instructions},
        {"role": "user", "content": test_items[2]["conversation_history"]}
    ]
)

print(completion.choices[0].message.content)

## Create the eval

An eval = a **data schema** (`data_source_config`) + **graders** (`testing_criteria`). We use a `string_check` grader with `eq` to compare the model output to `correct_label`.

In [ ]:
# Create the eval: define the data schema (conversation_history + correct_label)
# and a string_check grader that compares the model output to the correct label.

eval_obj = client.evals.create(
    name = "Recruiter Action Routing",

    data_source_config={
        "type": "custom",
        "item_schema": {
            "type": "object",
            "properties": {
                "conversation_history": {"type": "string"},
                "correct_label": {"type": "string"},
            },
            "required": ["conversation_history", "correct_label"],
        },
        "include_sample_schema": True
    },
    testing_criteria=[
        {
            "type": "string_check",
            "name": "Match output to human label",
            "input": "{{ sample.output_text }}",
            "operation": "eq",
            "reference": "{{ item.correct_label }}"
        }
    ],
)

print(eval_obj)
print(eval_obj.id)

## Upload the test JSONL

In [ ]:
# Upload the test JSONL file to OpenAI so the eval run can read it.

file = client.files.create(
    file=open("sms_conversations_test.jsonl", "rb"),
    purpose="evals"
)

print(file)
print(file.id)

## Create the eval run

In [ ]:
# Create the eval run: use gpt-4.1 with our instructions as the system prompt
# and the uploaded JSONL as the data source.

run = client.evals.runs.create(
    eval_obj.id, 

    name= "Recruiter text run",

    data_source={
        "type": "completions",
        "model": "gpt-4.1",
        "input_messages": {
            "type": "template",
            "template": [
                {"role": "developer", "content": instructions},
                {"role": "user", "content": "{{ item.conversation_history }}"}
            ],
        },
        "source": {"type": "file_id", "id": file.id},
    },
)

print(run)
print(run.id)

## Analyze the results

In [ ]:
# Retrieve the run and print its status (wait until it becomes 'completed').
run_retrieve = client.evals.runs.retrieve(
    eval_id=eval_obj.id, # YOUR_EVAL_ID
    run_id=run.id # YOUR_RUN_ID
    )


print(run_retrieve)
print(run_retrieve.status)

In [ ]:
# Print the pass / fail tallies from the completed run.
print(f"Errored = {run_retrieve.result_counts.errored}")
print(f"Total = {run_retrieve.result_counts.total}")
print(f"Passed = {run_retrieve.result_counts.passed}")
print(f"Failed = {run_retrieve.result_counts.failed}")

In [ ]:
# Fetch the per-item results: true labels come from the dataset,
# predicted labels come from the model's output (stripped of whitespace).

items = client.evals.runs.output_items.list(
    eval_id=eval_obj.id,
    run_id=run.id
)

y_true = [item.datasource_item['correct_label'] for item in items]
y_pred = [item.sample.output[0].content.strip() for item in items]

print(y_true)
print(y_pred)

In [ ]:
# Compute and print the model's overall accuracy against the human labels.
from sklearn.metrics import accuracy_score

print(f"accuracy_score: {accuracy_score(y_true, y_pred)}")

In [39]:
# Union of all seen labels, sorted:
labels = sorted(set(y_true) | set(y_pred))
print(labels)

['continue', 'end', 'schedule']


In [ ]:
# Build the confusion matrix and display it as a labeled pandas DataFrame.
from sklearn.metrics import confusion_matrix
import pandas as pd

cm = confusion_matrix(y_true, y_pred, labels=labels)

df_cm = pd.DataFrame(cm, index=labels, columns=labels)
print("Confusion Matrix:\n")
print(df_cm)

In [41]:
import numpy as np

for idx, label in enumerate(labels):
    TP = cm[idx, idx]
    FP = cm[:, idx].sum() - TP
    FN = cm[idx, :].sum() - TP
    TN = cm.sum() - (TP + FP + FN)
    print(f"\nLabel: {label}")
    print(f"  TP: {TP}")
    print(f"  FP: {FP}")
    print(f"  FN: {FN}")
    print(f"  TN: {TN}")


Label: continue
  TP: 25
  FP: 17
  FN: 0
  TN: 17

Label: end
  TP: 6
  FP: 1
  FN: 9
  TN: 43

Label: schedule
  TP: 1
  FP: 9
  FN: 18
  TN: 31


In [ ]:
# Plot the confusion matrix as a heatmap.

import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.heatmap(df_cm, annot=True, fmt='d', cmap="Blues")
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.title('Confusion Matrix')
plt.show()

---

## Bonus: Evaluate our actual multi-agent system

The cells above test a **single prompt** against `gpt-4.1` through OpenAI's cloud eval. That is useful as a baseline, but it doesn't test the code we actually wrote.

Below we run the **same test cases** through our real pipeline — `get_main_agent_response` — which routes through the Exit / Scheduling / Info advisors. Then we compute Accuracy + Confusion Matrix the same way.

In [ ]:
# Add the project root to sys.path so the `app` package is importable here.
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))   # adds the project root to sys.path

In [ ]:
# Import the LangChain LLM wrapper and our real multi-agent entry point.
from langchain_openai import ChatOpenAI
from app.modules.agents.main_agent import get_main_agent_response

In [ ]:
# Create the LLM our multi-agent system will use (deterministic: temperature=0).
llm = ChatOpenAI(model="gpt-4.1", temperature=0)

In [ ]:
# Run the same test cases through our real pipeline (get_main_agent_response),
# collecting the predicted action and the corresponding correct label for each.

y_pred_local = []
y_true_local = []

for item in test_items:
    action = get_main_agent_response(item["conversation_history"], llm)["action"]
    y_pred_local.append(action)
    y_true_local.append(item['correct_label'])

In [54]:
print("y_pred_local")
print(y_pred_local)
print("\n")

print("y_true_local")
print(y_true_local)

y_pred_local
['continue', 'continue', 'schedule', 'schedule', 'continue', 'continue', 'continue', 'schedule', 'end', 'continue', 'continue', 'continue', 'schedule', 'continue', 'continue', 'continue', 'schedule', 'continue', 'continue', 'end', 'continue', 'continue', 'schedule', 'continue', 'continue', 'schedule', 'schedule', 'continue', 'continue', 'schedule', 'end', 'end', 'continue', 'continue', 'continue', 'end', 'continue', 'continue', 'continue', 'schedule', 'continue', 'continue', 'continue', 'schedule', 'schedule', 'continue', 'continue', 'schedule', 'schedule', 'continue', 'continue', 'end', 'continue', 'continue', 'schedule', 'schedule', 'continue', 'continue', 'end']


y_true_local
['continue', 'schedule', 'schedule', 'end', 'continue', 'continue', 'continue', 'schedule', 'end', 'continue', 'continue', 'continue', 'end', 'continue', 'continue', 'continue', 'end', 'continue', 'schedule', 'end', 'continue', 'schedule', 'end', 'continue', 'schedule', 'schedule', 'end', 'continu

In [ ]:
# Print the multi-agent system's overall accuracy against the human labels.

from sklearn.metrics import accuracy_score

print(f"accuracy_score: {accuracy_score(y_true_local, y_pred_local)}")

In [ ]:
# Build the multi-agent system's confusion matrix and print it as a DataFrame.
from sklearn.metrics import confusion_matrix
import pandas as pd

labels = sorted(set(y_true_local) | set(y_pred_local))

cm = confusion_matrix(y_true_local, y_pred_local, labels=labels)

df_cm = pd.DataFrame(cm, index=labels, columns=labels)
print("Confusion Matrix:\n")
print(df_cm)


In [ ]:
# Plot the multi-agent system's confusion matrix as a heatmap.

import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.heatmap(df_cm, annot=True, fmt='d', cmap="Blues")
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.title('Multi-Agent System — Confusion Matrix')
plt.show()